# Explore Page Embeddings — Interactive 3D UMAP

Loads pre-computed pipeline pkl files, extracts embeddings, runs 3D UMAP, computes anomaly scores, and renders an interactive Plotly scatter where every point is a manuscript page with an inline thumbnail on hover.

## 1 — Import Required Libraries

In [ ]:
import sys
import os
import pickle
import base64
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import plotly.graph_objects as go

# Make sure the project root is on the Python path so `pipeline` can be imported
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.embeddings import (
    PageEmbedding,
    anomaly_scores,
    compute_umap_3d,
    load_embeddings_from_folder,
)

print("Imports OK — project root:", PROJECT_ROOT)

## 2 — Configure Input Folder and Discover Results

In [ ]:
# ── CONFIGURE THIS ─────────────────────────────────────────────────────────────
IMAGES_FOLDER = Path("../data/all_images")   # <-- change to your images folder
IMAGE_EXTS    = (".jpg", ".jpeg", ".png", ".tif", ".tiff")
# ───────────────────────────────────────────────────────────────────────────────

IMAGES_FOLDER = IMAGES_FOLDER.resolve()
results_dir   = IMAGES_FOLDER / "results"

assert IMAGES_FOLDER.is_dir(), f"Folder not found: {IMAGES_FOLDER}"
assert results_dir.is_dir(),   f"results/ subfolder not found in {IMAGES_FOLDER}"

image_files = sorted(p for p in IMAGES_FOLDER.iterdir() if p.suffix.lower() in IMAGE_EXTS)
pkl_files   = [results_dir / img.with_suffix(".pkl").name for img in image_files]
found_pkls  = [p for p in pkl_files if p.is_file()]

print(f"Images folder : {IMAGES_FOLDER}")
print(f"Images found  : {len(image_files)}")
print(f"PKL files found: {len(found_pkls)} / {len(image_files)}")

## 3 — Load Pickle Files and Extract Embeddings

In [ ]:
file = "/Users/luissalamanca/Dropbox/My_stuff/05_SDSCresearch/10_SideProjects/00_MedievalCambridge/line_counting/data/all_images/results/MS-GG-00001-00001-000-01274.pkl"
with open(file, "rb") as f:
    data = pickle.load(f)

In [ ]:
records = []   # list of dicts, one per successfully loaded image

for img_path in image_files:
    pkl_path = results_dir / img_path.with_suffix(".pkl").name
    if not pkl_path.is_file():
        continue
    with open(pkl_path, "rb") as fh:
        data = pickle.load(fh)

    emb = data.get("embedding")   # PageEmbedding or None
    if emb is None:
        print(f"  [SKIP] {img_path.name} — no embedding in pkl")
        continue

    records.append({
        "filename"  : img_path.name,
        "img_path"  : str(img_path),
        "embedding" : emb,
        # convenience flags
        "has_structural" : emb.structural_vec.size > 0,
        "has_vit_rgb"    : emb.vit_rgb_vec.size   > 0,
        "has_vit_mask"   : emb.vit_mask_vec.size  > 0,
    })

print(f"\nLoaded {len(records)} embeddings.")

## 4 — Report Available Embeddings and Select Active Ones

Set the flags below to choose which embedding components to use when building the UMAP feature matrix.

In [ ]:
n_total       = len(records)
n_structural  = sum(r["has_structural"] for r in records)
n_vit_rgb     = sum(r["has_vit_rgb"]    for r in records)
n_vit_mask    = sum(r["has_vit_mask"]   for r in records)

print("Embedding type     | count / total")
print("-------------------|--------------")
print(f"structural_vec     | {n_structural} / {n_total}")
print(f"vit_rgb_vec        | {n_vit_rgb}    / {n_total}")
print(f"vit_mask_vec       | {n_vit_mask}   / {n_total}")

# ── SELECT WHICH EMBEDDING TYPES TO USE ────────────────────────────────────────
USE_STRUCTURAL = True    # 24-d normalised structural features
USE_VIT_RGB    = True    # 768-d ViT CLS from deskewed BGR image
USE_VIT_MASK   = False   # 768-d ViT CLS from text+figure overlay
# ───────────────────────────────────────────────────────────────────────────────

# Filter to records that have all required embedding types
def _record_has_required(r):
    ok = True
    if USE_STRUCTURAL and not r["has_structural"]: ok = False
    if USE_VIT_RGB    and not r["has_vit_rgb"]:    ok = False
    if USE_VIT_MASK   and not r["has_vit_mask"]:   ok = False
    return ok

active_records = [r for r in records if _record_has_required(r)]
print(f"\nRecords with all required embeddings: {len(active_records)} / {n_total}")

if len(active_records) < 4:
    print("WARNING: fewer than 4 samples — UMAP may not be meaningful.")

# Rebuild combined_vec from selected sub-vectors (so UMAP sees only the chosen ones)
for r in active_records:
    emb = r["embedding"]
    parts = []
    if USE_STRUCTURAL: parts.append(emb.structural_vec)
    if USE_VIT_RGB:    parts.append(emb.vit_rgb_vec)
    if USE_VIT_MASK:   parts.append(emb.vit_mask_vec)
    emb.combined_vec = np.concatenate(parts)

## 5 — Run UMAP for 3D Projection

In [ ]:
UMAP_N_NEIGHBORS = 10    # local neighbourhood size
UMAP_MIN_DIST    = 0.1   # minimum distance in low-d space

embeddings_list = [r["embedding"] for r in active_records]

print(f"Running 3D UMAP on {len(embeddings_list)} samples …")
compute_umap_3d(embeddings_list, n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST)
print("Done.")

## 6 — Compute Anomaly Scores

In [ ]:
anomaly_scores(embeddings_list)   # sets .anomaly_score on each embedding in-place

scores = [r["embedding"].anomaly_score for r in active_records]
df_scores = pd.DataFrame({
    "filename"     : [r["filename"] for r in active_records],
    "anomaly_score": scores,
}).sort_values("anomaly_score", ascending=False)

print("Top-10 most anomalous pages:")
display(df_scores.head(10))

## 7 — Build Interactive 3D Plot with Hover Previews

Each point = one page. Colour encodes anomaly score (blue → red). Hover shows an inline image thumbnail, the anomaly score, and key structural features.

In [ ]:
import json

THUMB_MAX_PX  = 120   # max thumbnail side in pixels
THUMB_QUALITY = 65    # JPEG quality

# ── Structural feature names (same order as compute_structural_features) ────
STRUCT_NAMES = [
    "binding_left", "binding_right", "deskew_angle",
    "margin_width/W", "top_margin/H", "ruler_height/H",
    "bottom_margin/H", "binding_width/W", "H/W",
    "n_lines/H", "text_coverage", "ink_retention",
    "fig_norm", "fig_coverage", "fig_mean_area", "fig_std_area",
    "post_n_lines/H", "removed_corner/seg_n", "removed_narrow/seg_n",
    "post_fig_norm", "fig_rm_corner/seg_n", "fig_rm_narrow/seg_n",
    "is_double_col", "gutter_x/W",
]


def _make_thumb_b64(img_path: str,
                    max_px: int = THUMB_MAX_PX,
                    quality: int = THUMB_QUALITY) -> str:
    """Resize image, encode as JPEG base64 data URI. Returns '' on failure."""
    img = cv2.imread(img_path)
    if img is None:
        return ""
    h, w = img.shape[:2]
    scale = max_px / max(h, w)
    if scale < 1.0:
        img = cv2.resize(img, (int(w * scale), int(h * scale)),
                         interpolation=cv2.INTER_AREA)
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        return ""
    b64 = base64.b64encode(buf.tobytes()).decode("ascii")
    return "data:image/jpeg;base64," + b64


# ── Build per-point data ─────────────────────────────────────────────────────
xs, ys, zs   = [], [], []
filenames    = []
anomaly_list = []
thumbnails   = []   # base64 data URIs

for r in active_records:
    emb = r["embedding"]
    x, y, z = emb.umap_xyz
    xs.append(x); ys.append(y); zs.append(z)
    filenames.append(r["filename"])
    anomaly_list.append(emb.anomaly_score)
    thumbnails.append(_make_thumb_b64(r["img_path"]))

# ── Write thumbs_data.js  ────────────────────────────────────────────────────
# This is a separate file so the HTML itself stays small and renders the
# Plotly chart immediately. The browser loads thumbs_data.js in the background.
thumbs_js_path = results_dir / "thumbs_data.js"
thumb_map = {filenames[i]: thumbnails[i] for i in range(len(filenames))}
with open(thumbs_js_path, "w", encoding="utf-8") as fh:
    fh.write("var THUMB_DATA = ")
    json.dump(thumb_map, fh)
    fh.write(";\n")

size_mb = thumbs_js_path.stat().st_size / 1e6
print(f"Saved {len(thumbnails)} thumbnails → {thumbs_js_path.name}  ({size_mb:.1f} MB)")

In [ ]:
OUTPUT_HTML = str(results_dir / "umap_3d_explore.html")

# ── Per-point metadata (NO thumbnail data — loaded from thumbs_data.js) ─────
js_data = []
for i, r in enumerate(active_records):
    emb_i = r["embedding"]
    feats = {}
    if emb_i.structural_vec.size > 0:
        for j, name in enumerate(STRUCT_NAMES[:len(emb_i.structural_vec)]):
            feats[name] = round(float(emb_i.structural_vec[j]), 4)
    js_data.append({
        "filename" : r["filename"],
        "anomaly"  : round(float(emb_i.anomaly_score), 4),
        "features" : feats,
    })

# ── Plotly figure ────────────────────────────────────────────────────────────
fig_clean = go.Figure()
fig_clean.add_trace(go.Scatter3d(
    x=xs, y=ys, z=zs,
    mode="markers",
    marker=dict(
        size=6,
        color=anomaly_list,
        colorscale="RdBu_r",
        colorbar=dict(title="Anomaly<br>score", thickness=16, len=0.6),
        cmin=0.0, cmax=1.0,
        opacity=0.85,
    ),
    text=filenames,
    hovertemplate="<b>%{text}</b><br>click to inspect<extra></extra>",
))
fig_clean.update_layout(
    title=dict(text="Page Embeddings — 3D UMAP  (click a point to inspect)", font=dict(size=16)),
    scene=dict(xaxis_title="UMAP-1", yaxis_title="UMAP-2", zaxis_title="UMAP-3"),
    margin=dict(l=0, r=250, b=0, t=36),
    height=700,
)

fig_json     = fig_clean.to_json()
js_data_json = json.dumps(js_data)

# ── Build standalone HTML ────────────────────────────────────────────────────
# thumbs_data.js must sit in the same folder as this HTML (results/).
# Open with any browser — no server needed (base64 data, no CORS).
html = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>Page Embeddings — 3D UMAP</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <!-- thumbnail data (base64 JPEGs) loaded separately so the page renders first -->
  <script src="thumbs_data.js"></script>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{ display: flex; height: 100vh; font-family: sans-serif; font-size: 13px; background: #fff; }}
    #plot {{ flex: 1; min-width: 0; }}
    #sidebar {{
      width: 250px; min-width: 250px; padding: 12px;
      border-left: 1px solid #ddd; overflow-y: auto;
      background: #fafafa;
    }}
    #sidebar img {{ width: 100%; border: 1px solid #ccc; margin-bottom: 8px; display: block; }}
    #sidebar h3  {{ font-size: 13px; word-break: break-all; margin-bottom: 6px; }}
    #sidebar .score {{ margin-bottom: 10px; }}
    #sidebar table {{ width: 100%; border-collapse: collapse; font-size: 11px; }}
    #sidebar td {{ padding: 2px 4px; border-bottom: 1px solid #eee; vertical-align: top; }}
    #sidebar td:first-child {{ color: #888; width: 58%; }}
    #no-click {{ color: #aaa; padding-top: 8px; }}
    #detail   {{ display: none; }}
  </style>
</head>
<body>
  <div id="plot"></div>
  <div id="sidebar">
    <div id="no-click">Click a point to inspect it</div>
    <div id="detail">
      <img id="thumb" src="" alt="page thumbnail">
      <h3 id="fname"></h3>
      <p class="score">Anomaly score: <b id="ascore"></b></p>
      <table id="feat-table"></table>
    </div>
  </div>

  <script>
    var figData   = {fig_json};
    var pointData = {js_data_json};

    // plotly_click does NOT interfere with 3D camera drag — plotly_hover does.
    Plotly.newPlot('plot', figData.data, figData.layout,
                   {{responsive: true, scrollZoom: true}});

    document.getElementById('plot').on('plotly_click', function(evt) {{
      var idx = evt.points[0].pointIndex;
      var d   = pointData[idx];
      document.getElementById('no-click').style.display = 'none';
      document.getElementById('detail').style.display   = 'block';

      // Thumbnail: look up in THUMB_DATA if available
      var src = (typeof THUMB_DATA !== 'undefined' && THUMB_DATA[d.filename])
                ? THUMB_DATA[d.filename] : '';
      document.getElementById('thumb').src          = src;
      document.getElementById('fname').textContent  = d.filename;
      document.getElementById('ascore').textContent = d.anomaly.toFixed(4);

      var rows = '';
      for (var k in d.features) {{
        rows += '<tr><td>' + k + '</td><td>' + d.features[k].toFixed(4) + '</td></tr>';
      }}
      document.getElementById('feat-table').innerHTML = rows;
    }});
  </script>
</body>
</html>
"""

with open(OUTPUT_HTML, "w", encoding="utf-8") as fh:
    fh.write(html)

html_mb = Path(OUTPUT_HTML).stat().st_size / 1e6
print(f"Saved HTML : {OUTPUT_HTML}  ({html_mb:.2f} MB)")
print(f"Also needs : {results_dir}/thumbs_data.js  (keep in same folder)")

from IPython.display import IFrame
IFrame(src=OUTPUT_HTML, width="100%", height=720)